<a href="https://colab.research.google.com/github/iang-br/Modelagem-de-dados-com-Redes-Neurais-Artificiais/blob/main/C%C3%B3pia_de_Cats_and_Dogs_Analise_Exploratoria.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import os
import urllib.request
from pathlib import Path
from collections import Counter
import zipfile
import pandas as pd
from IPython.display import display

# Caminho local temporário para o arquivo ZIP
arquivo_zip = Path("/tmp/kagglecatsanddogs_5340.zip")

# Se o arquivo não existir localmente, faz o download direto (como no notebook de referência)
if not arquivo_zip.exists():
    print("Baixando o dataset original (Cats vs Dogs)... isso pode levar alguns minutos.")
    url = "https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip"
    urllib.request.urlretrieve(url, arquivo_zip)
    print("Download concluído!")

print("Arquivo localizado:", arquivo_zip)

extensoes_imagem = {
    ".jpg", ".jpeg", ".png", ".bmp", ".gif", ".webp", ".tif", ".tiff"
}

with zipfile.ZipFile(arquivo_zip) as zip_ref:
    arquivos = [
        item for item in zip_ref.infolist()
        if not item.is_dir()
    ]

    imagens = [
        item for item in arquivos
        if Path(item.filename).suffix.lower() in extensoes_imagem
    ]

    contagem_pastas = Counter(
        str(Path(item.filename).parent) for item in imagens
    )

    tabela = pd.DataFrame(
        contagem_pastas.items(),
        columns=["Pasta dentro do ZIP", "Quantidade de imagens"]
    ).sort_values("Pasta dentro do ZIP")

    print(f"\nTotal de arquivos: {len(arquivos):,}")
    print(f"Arquivos com extensão de imagem: {len(imagens):,}")
    print(f"Outros arquivos: {len(arquivos) - len(imagens):,}")

    tamanho_total = sum(item.file_size for item in arquivos)
    print(f"Tamanho descompactado: {tamanho_total / 1024**3:.2f} GiB")

    print("\nIMAGENS POR PASTA")
    display(tabela.reset_index(drop=True))

    print("\nPRIMEIROS 10 NOMES DE IMAGENS")
    for item in imagens[:10]:
        print(item.filename)

Baixando o dataset original (Cats vs Dogs)... isso pode levar alguns minutos.
Download concluído!
Arquivo localizado: /tmp/kagglecatsanddogs_5340.zip

Total de arquivos: 25,004
Arquivos com extensão de imagem: 25,000
Outros arquivos: 4
Tamanho descompactado: 0.81 GiB

IMAGENS POR PASTA


,Pasta dentro do ZIP,Quantidade de imagens
0,PetImages/Cat,12500
1,PetImages/Dog,12500



PRIMEIROS 10 NOMES DE IMAGENS
PetImages/Cat/0.jpg
PetImages/Cat/1.jpg
PetImages/Cat/10.jpg
PetImages/Cat/100.jpg
PetImages/Cat/1000.jpg
PetImages/Cat/10000.jpg
PetImages/Cat/10001.jpg
PetImages/Cat/10002.jpg
PetImages/Cat/10003.jpg
PetImages/Cat/10004.jpg


##Tudo

In [ ]:
import random
import zipfile
from pathlib import PurePosixPath
from PIL import Image
import matplotlib.pyplot as plt

# Usa o caminho arquivo_zip encontrado na célula anterior.
sorteio = random.Random(42)

fig, eixos = plt.subplots(2, 5, figsize=(15, 7))
falhas = []

with zipfile.ZipFile(arquivo_zip) as zip_ref:
    for linha, classe in enumerate(["Cat", "Dog"]):

        # Lista as imagens da classe.
        candidatos = [
            nome for nome in zip_ref.namelist()
            if PurePosixPath(nome).parent.as_posix()
            == f"PetImages/{classe}"
            and PurePosixPath(nome).suffix.lower()
            in {".jpg", ".jpeg", ".png"}
        ]

        sorteio.shuffle(candidatos)
        coluna = 0

        for nome in candidatos:
            if coluna == 5:
                break

            try:
                with zip_ref.open(nome) as arquivo:
                    with Image.open(arquivo) as imagem:
                        imagem.load()  # Testa a decodificação desta imagem.
                        largura, altura = imagem.size
                        modo = imagem.mode
                        visualizacao = imagem.convert("RGB")

                ax = eixos[linha, coluna]
                ax.imshow(visualizacao)
                ax.set_title(
                    f"{classe}: {PurePosixPath(nome).name}\n"
                    f"{largura} × {altura} pixels | {modo}"
                )
                ax.axis("off")
                coluna += 1

            except Exception as erro:
                falhas.append({
                    "arquivo": nome,
                    "erro": str(erro)
                })

        # Oculta espaços caso não haja cinco imagens legíveis.
        for restante in range(coluna, 5):
            eixos[linha, restante].axis("off")

fig.suptitle("Amostra aleatória: gatos e cachorros", fontsize=16)
plt.tight_layout()
plt.show()

print("Falhas encontradas durante a seleção:", len(falhas))
for falha in falhas:
    print(falha)

In [ ]:
import zipfile
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import PurePosixPath
from PIL import Image, ImageFile
from tqdm.auto import tqdm
from IPython.display import display

# Não aceitar silenciosamente imagens truncadas.
ImageFile.LOAD_TRUNCATED_IMAGES = False

registros = []

with zipfile.ZipFile(arquivo_zip) as zip_ref:

    nomes = [
        nome for nome in zip_ref.namelist()
        if PurePosixPath(nome).parent.as_posix()
        in {"PetImages/Cat", "PetImages/Dog"}
        and PurePosixPath(nome).suffix.lower()
        in {".jpg", ".jpeg", ".png", ".bmp", ".gif",
            ".webp", ".tif", ".tiff"}
    ]

    for nome in tqdm(nomes, desc="Analisando imagens"):
        registro = {
            "arquivo": nome,
            "classe": PurePosixPath(nome).parent.name,
            "largura": None,
            "altura": None,
            "modo_cor": None,
            "status": "erro",
            "erro": ""
        }

        try:
            with zip_ref.open(nome) as arquivo:
                with Image.open(arquivo) as imagem:
                    imagem.load()  # Decodifica a imagem.
                    registro.update({
                        "largura": imagem.width,
                        "altura": imagem.height,
                        "modo_cor": imagem.mode,
                        "status": "ok"
                    })

        except Exception as erro:
            registro["erro"] = str(erro)

        registros.append(registro)

diagnostico = pd.DataFrame(registros)
validas = diagnostico[diagnostico["status"] == "ok"].copy()
falhas = diagnostico[diagnostico["status"] == "erro"].copy()

print("RESUMO POR CLASSE")
display(pd.crosstab(
    diagnostico["classe"],
    diagnostico["status"]
))

print("\nMODOS DE COR DAS IMAGENS LEGÍVEIS")
display(pd.crosstab(
    validas["classe"],
    validas["modo_cor"]
))

print("\nESTATÍSTICAS DAS DIMENSÕES — PIXELS")
display(
    validas[["largura", "altura"]]
    .astype(float)
    .describe()
    .round(1)
)

print(f"\nImagens com falha de leitura: {len(falhas)}")
display(falhas[["arquivo", "erro"]].head(20))

# Distribuições de largura e altura por classe.
fig, eixos = plt.subplots(1, 2, figsize=(12, 4))

for classe, cor in [("Cat", "#00429d"), ("Dog", "#2ca25f")]:
    dados = validas[validas["classe"] == classe]

    eixos[0].hist(
        dados["largura"].astype(float),
        bins=40, alpha=0.5, label=classe, color=cor
    )
    eixos[1].hist(
        dados["altura"].astype(float),
        bins=40, alpha=0.5, label=classe, color=cor
    )

for ax, dimensao in zip(eixos, ["Largura", "Altura"]):
    ax.set_title(f"Distribuição de {dimensao.lower()}")
    ax.set_xlabel(f"{dimensao} em pixels")
    ax.set_ylabel("Quantidade de imagens")
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Utiliza a tabela "validas" criada na análise anterior.
analise = validas.copy()

analise["largura"] = analise["largura"].astype(int)
analise["altura"] = analise["altura"].astype(int)

analise["menor_lado"] = analise[
    ["largura", "altura"]
].min(axis=1)

analise["maior_lado"] = analise[
    ["largura", "altura"]
].max(axis=1)

analise["proporcao_extrema"] = (
    analise["maior_lado"] / analise["menor_lado"]
)

# Critérios exploratórios para selecionar casos para inspeção.
# Não são regras automáticas de exclusão.
suspeitas = analise[
    (analise["menor_lado"] < 64)
    | (analise["proporcao_extrema"] > 4)
].sort_values(["menor_lado", "proporcao_extrema"])

print("Imagens selecionadas para inspeção:", len(suspeitas))

display(
    suspeitas[
        ["arquivo", "classe", "largura", "altura",
         "modo_cor", "proporcao_extrema"]
    ].head(30)
)

print("\nQuantidade por classe:")
print(suspeitas["classe"].value_counts())

# Exibir até 12 exemplos.
amostra = suspeitas.head(12)

if not amostra.empty:
    fig, eixos = plt.subplots(3, 4, figsize=(14, 10))

    for ax in eixos.flat:
        ax.axis("off")

    with zipfile.ZipFile(arquivo_zip) as zip_ref:
        for ax, (_, linha) in zip(eixos.flat, amostra.iterrows()):
            with zip_ref.open(linha["arquivo"]) as arquivo:
                with Image.open(arquivo) as imagem:
                    ax.imshow(imagem.convert("RGB"))

            ax.set_title(
                f"{linha['arquivo']}\n"
                f"{linha['largura']} × {linha['altura']} pixels",
                fontsize=9
            )

    plt.tight_layout()
    plt.show()
else:
    print("Nenhuma imagem atende aos critérios de inspeção.")

In [ ]:
# Exibe todas as imagens selecionadas, em grupos de até 12.
with zipfile.ZipFile(arquivo_zip) as zip_ref:

    for inicio in range(0, len(suspeitas), 12):
        grupo = suspeitas.iloc[inicio:inicio + 12]

        fig, eixos = plt.subplots(3, 4, figsize=(14, 10))

        for ax in eixos.flat:
            ax.axis("off")

        for ax, (_, linha) in zip(eixos.flat, grupo.iterrows()):
            with zip_ref.open(linha["arquivo"]) as arquivo:
                with Image.open(arquivo) as imagem:
                    ax.imshow(
                        imagem.convert("RGB"),
                        interpolation="nearest"
                    )

            ax.set_title(
                f"{linha['arquivo']}\n"
                f"{linha['largura']} × {linha['altura']} pixels",
                fontsize=9
            )

        fig.suptitle(
            f"Inspeção: imagens {inicio + 1} "
            f"a {inicio + len(grupo)}",
            fontsize=14
        )
        plt.tight_layout()
        plt.show()

In [ ]:
exclusoes_confirmadas = {
    "PetImages/Cat/666.jpg": "Falha de leitura",
    "PetImages/Dog/11702.jpg": "Falha de leitura",
    "PetImages/Cat/5673.jpg": "Resolução de apenas 4 × 4 pixels",
    "PetImages/Dog/4367.jpg": "Logotipo Yahoo! Mail",
    "PetImages/Cat/9171.jpg": "Imagem de uma rosa"
}

In [ ]:
tabela_exclusoes = pd.DataFrame(
    exclusoes_confirmadas.items(),
    columns=["Arquivo", "Motivo"]
)

display(tabela_exclusoes)

In [ ]:
# Última linha, segunda coluna do painel 25 a 36
registro = suspeitas.iloc[33]

print("Arquivo:", registro["arquivo"])
print("Dimensões:", registro["largura"], "×", registro["altura"])

with zipfile.ZipFile(arquivo_zip) as zip_ref:
    with zip_ref.open(registro["arquivo"]) as arquivo:
        with Image.open(arquivo) as imagem:
            plt.figure(figsize=(6, 6))
            plt.imshow(imagem.convert("RGB"), interpolation="nearest")
            plt.title(registro["arquivo"])
            plt.axis("off")
            plt.show()

In [ ]:
# Exibe todas as imagens selecionadas, em grupos de até 12.
with zipfile.ZipFile(arquivo_zip) as zip_ref:

    for inicio in[36]:
        grupo = suspeitas.iloc[inicio:inicio + 12]

        fig, eixos = plt.subplots(3, 4, figsize=(14, 10))

        for ax in eixos.flat:
            ax.axis("off")

        for ax, (_, linha) in zip(eixos.flat, grupo.iterrows()):
            with zip_ref.open(linha["arquivo"]) as arquivo:
                with Image.open(arquivo) as imagem:
                    ax.imshow(
                        imagem.convert("RGB"),
                        interpolation="nearest"
                    )

            ax.set_title(
                f"{linha['arquivo']}\n"
                f"{linha['largura']} × {linha['altura']} pixels",
                fontsize=9
            )

        fig.suptitle(
            f"Inspeção: imagens {inicio + 1} "
            f"a {inicio + len(grupo)}",
            fontsize=14
        )
        plt.tight_layout()
        plt.show()

In [ ]:
import hashlib
from collections import defaultdict

grupos_hash = defaultdict(list)

with zipfile.ZipFile(arquivo_zip) as zip_ref:
    for nome in tqdm(validas["arquivo"], desc="Buscando duplicados"):
        with zip_ref.open(nome) as arquivo:
            hash_arquivo = hashlib.file_digest(
                arquivo, "sha256"
            ).hexdigest()

        grupos_hash[hash_arquivo].append(nome)

duplicados = [
    nomes for nomes in grupos_hash.values()
    if len(nomes) > 1
]

print("Grupos de arquivos idênticos:", len(duplicados))
print(
    "Cópias extras:",
    sum(len(grupo) - 1 for grupo in duplicados)
)

for numero, grupo in enumerate(duplicados[:20], start=1):
    print(f"\nGrupo {numero}:")
    for nome in grupo:
        print(" ", nome)

In [ ]:
from pathlib import PurePosixPath

# Seleciona grupos com arquivos presentes em classes diferentes.
conflitos = [
    grupo for grupo in duplicados
    if len({
        PurePosixPath(nome).parent.name
        for nome in grupo
    }) > 1
]

print("Grupos com conflito de classe:", len(conflitos))

with zipfile.ZipFile(arquivo_zip) as zip_ref:
    for numero, grupo in enumerate(conflitos, start=1):
        print(f"\nCONFLITO {numero}")
        for nome in grupo:
            print(" ", nome)

        # Basta abrir uma cópia: os arquivos são idênticos.
        with zip_ref.open(grupo[0]) as arquivo:
            with Image.open(arquivo) as imagem:
                plt.figure(figsize=(5, 5))
                plt.imshow(imagem.convert("RGB"))
                plt.title(f"Conflito {numero}")
                plt.axis("off")
                plt.show()

In [ ]:
novas_exclusoes = {
    "PetImages/Cat/11565.jpg": "Imagem substituta: PHOTO UNAVAILABLE",
    "PetImages/Dog/10401.jpg": "Imagem substituta: PHOTO UNAVAILABLE",
    "PetImages/Dog/10797.jpg": "Imagem substituta: PHOTO UNAVAILABLE",
    "PetImages/Dog/2877.jpg": "Imagem substituta: PHOTO UNAVAILABLE",
    "PetImages/Cat/8456.jpg": "Imagem substituta: texto Adopted",
    "PetImages/Dog/8736.jpg": "Imagem substituta: texto Adopted"
}

exclusoes_confirmadas.update(novas_exclusoes)

display(pd.DataFrame(
    exclusoes_confirmadas.items(),
    columns=["Arquivo", "Motivo"]
))

print("Total de exclusões registradas:", len(exclusoes_confirmadas))

In [ ]:
# Imagens que ainda precisam de revisão visual.
pendentes = {
    "PetImages/Cat/835.jpg",
    "PetImages/Dog/10747.jpg"
}

# Começa pelas imagens legíveis, sem as exclusões confirmadas.
candidatas = validas[
    ~validas["arquivo"].isin(exclusoes_confirmadas)
].copy()

# Seleciona uma cópia por grupo de arquivos idênticos.
# A ordenação torna a escolha reproduzível.
nomes_candidatos = set(candidatas["arquivo"])
copias_extras = set()

for grupo in duplicados:
    restantes = sorted(
        nome for nome in grupo
        if nome in nomes_candidatos
    )
    copias_extras.update(restantes[1:])

# Lista após retirar as repetições.
sem_duplicatas = candidatas[
    ~candidatas["arquivo"].isin(copias_extras)
].copy()

# Separa os casos pendentes.
base_provisoria = sem_duplicatas[
    ~sem_duplicatas["arquivo"].isin(pendentes)
].copy()

print("Imagens legíveis:", len(validas))
print("Após exclusões confirmadas:", len(candidatas))
print("Cópias extras retiradas da lista:", len(copias_extras))
print("Após retirar duplicatas:", len(sem_duplicatas))
print("Base provisória, sem pendentes:", len(base_provisoria))

print("\nQUANTIDADE POR CLASSE")
display(
    base_provisoria.groupby("classe")
    .size()
    .rename("Quantidade")
    .to_frame()
)

In [ ]:
import zipfile
import matplotlib.pyplot as plt
from PIL import Image

arquivos_pendentes = [
    "PetImages/Cat/835.jpg",
    "PetImages/Dog/10747.jpg"
]

fig, eixos = plt.subplots(1, 2, figsize=(12, 6))

with zipfile.ZipFile(arquivo_zip) as zip_ref:
    for ax, nome in zip(eixos, arquivos_pendentes):
        with zip_ref.open(nome) as arquivo:
            with Image.open(arquivo) as imagem:
                imagem.load()
                largura, altura = imagem.size

                ax.imshow(
                    imagem.convert("RGB"),
                    interpolation="nearest"
                )

        ax.set_title(f"{nome}\n{largura} × {altura} pixels")
        ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
exclusoes_confirmadas.update({
    "PetImages/Cat/835.jpg":
        "Imagem aparentemente branca, sem animal visível",
    "PetImages/Dog/10747.jpg":
        "Animal não reconhecível na inspeção visual"
})

# Recria a base a partir das imagens legíveis,
# retirando exclusões e cópias extras já identificadas.
base_provisoria = validas[
    ~validas["arquivo"].isin(exclusoes_confirmadas)
    & ~validas["arquivo"].isin(copias_extras)
].copy()

pendentes = set()

print("Exclusões por leitura ou conteúdo:",
      len(exclusoes_confirmadas))
print("Imagens na base provisória:", len(base_provisoria))

display(
    base_provisoria.groupby("classe")
    .size()
    .rename("Quantidade")
    .to_frame()
)

In [ ]:
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageOps
from scipy.fft import dctn
from tqdm.auto import tqdm
from IPython.display import display

# ======================================================
# 1. CALCULAR UM RESUMO VISUAL DE 64 BITS POR IMAGEM
# ======================================================
def calcular_phash(imagem):
    imagem = ImageOps.exif_transpose(imagem)
    cinza = imagem.convert("L").resize(
        (32, 32), Image.Resampling.LANCZOS
    )

    matriz = np.asarray(cinza, dtype=float)
    frequencias = dctn(matriz, type=2, norm="ortho")[:8, :8]

    # A primeira frequência representa principalmente o brilho.
    mediana = np.median(frequencias.flatten()[1:])
    bits = (frequencias > mediana).flatten()
    bits[0] = False

    valor = 0
    for bit in bits:
        valor = (valor << 1) | int(bit)

    return valor


hashes_visuais = []
falhas_phash = []

with zipfile.ZipFile(arquivo_zip) as zip_ref:
    for nome in tqdm(
        base_provisoria["arquivo"],
        desc="Calculando resumos visuais"
    ):
        try:
            with zip_ref.open(nome) as arquivo:
                with Image.open(arquivo) as imagem:
                    imagem.load()
                    resumo = calcular_phash(imagem)

            hashes_visuais.append((nome, resumo))

        except Exception as erro:
            falhas_phash.append((nome, str(erro)))

# ======================================================
# 2. BUSCAR HASHES PRÓXIMOS COM UMA ÁRVORE DE DISTÂNCIAS
# ======================================================
def distancia(a, b):
    # Quantidade de bits diferentes.
    return (a ^ b).bit_count()


arvore = None

def inserir(valor, indice):
    global arvore

    if arvore is None:
        arvore = [valor, [indice], {}]
        return

    no = arvore

    while True:
        d = distancia(valor, no[0])

        if d == 0:
            no[1].append(indice)
            return

        if d not in no[2]:
            no[2][d] = [valor, [indice], {}]
            return

        no = no[2][d]


def buscar(valor, limite):
    encontrados = []
    pilha = [] if arvore is None else [arvore]

    while pilha:
        no = pilha.pop()
        d = distancia(valor, no[0])

        if d <= limite:
            encontrados.extend((indice, d) for indice in no[1])

        for aresta, filho in no[2].items():
            if d - limite <= aresta <= d + limite:
                pilha.append(filho)

    return encontrados


# Critério exploratório: até 4 bits diferentes em 64.
# Menor distância significa hashes mais próximos.
limite = 4
pares = []

for indice, (nome, resumo) in enumerate(
    tqdm(hashes_visuais, desc="Comparando resumos")
):
    for anterior, d in buscar(resumo, limite):
        outro_nome = hashes_visuais[anterior][0]

        pares.append({
            "Imagem A": outro_nome,
            "Imagem B": nome,
            "Distância pHash": d,
            "Classes diferentes":
                outro_nome.split("/")[-2] != nome.split("/")[-2]
        })

    inserir(resumo, indice)

pares_semelhantes = pd.DataFrame(
    pares,
    columns=[
        "Imagem A", "Imagem B",
        "Distância pHash", "Classes diferentes"
    ]
).sort_values(
    ["Distância pHash", "Imagem A", "Imagem B"]
).reset_index(drop=True)

print("\nPares candidatos:", len(pares_semelhantes))
print("Falhas nesta análise:", len(falhas_phash))
display(pares_semelhantes.head(20))

# ======================================================
# 3. MOSTRAR OS PRIMEIROS CINCO PARES PARA REVISÃO
# ======================================================
with zipfile.ZipFile(arquivo_zip) as zip_ref:
    for numero, linha in pares_semelhantes.head(5).iterrows():
        fig, eixos = plt.subplots(1, 2, figsize=(10, 4))

        for ax, coluna in zip(eixos, ["Imagem A", "Imagem B"]):
            nome = linha[coluna]

            with zip_ref.open(nome) as arquivo:
                with Image.open(arquivo) as imagem:
                    visual = ImageOps.exif_transpose(imagem)
                    ax.imshow(visual.convert("RGB"))

            ax.set_title(nome, fontsize=10)
            ax.axis("off")

        fig.suptitle(
            f"Par {numero + 1} — "
            f"distância pHash: {linha['Distância pHash']}"
        )
        plt.tight_layout()
        plt.show()

if pares_semelhantes.empty:
    print("Nenhum par encontrado com esse limite.")

In [ ]:
# Registra os cinco pares que acabamos de revisar.
pares_confirmados = pares_semelhantes.head(5).copy()

print("Pares confirmados visualmente:", len(pares_confirmados))
print("Total de pares candidatos:", len(pares_semelhantes))
print("Falhas na análise:", len(falhas_phash))

display(pares_confirmados)

In [ ]:
# Mostra o sexto e último par candidato.
par = pares_semelhantes.iloc[5]

print("Distância pHash:", par["Distância pHash"])
print("Classes diferentes:", par["Classes diferentes"])

fig, eixos = plt.subplots(1, 2, figsize=(12, 5))

with zipfile.ZipFile(arquivo_zip) as zip_ref:
    for ax, coluna in zip(eixos, ["Imagem A", "Imagem B"]):
        nome = par[coluna]

        with zip_ref.open(nome) as arquivo:
            with Image.open(arquivo) as imagem:
                visual = ImageOps.exif_transpose(imagem)
                largura, altura = visual.size
                ax.imshow(visual.convert("RGB"))

        ax.set_title(f"{nome}\n{largura} × {altura} pixels")
        ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Os seis pares foram revisados visualmente.
pares_confirmados = pares_semelhantes.copy()

dimensoes = validas.set_index("arquivo")
duplicatas_visuais = set()
decisoes = []

for _, par in pares_confirmados.iterrows():
    nomes = [par["Imagem A"], par["Imagem B"]]

    # Maior resolução primeiro; nome como desempate.
    nomes = sorted(
        nomes,
        key=lambda nome: (
            -int(dimensoes.loc[nome, "largura"])
            * int(dimensoes.loc[nome, "altura"]),
            nome
        )
    )

    manter, retirar = nomes
    duplicatas_visuais.add(retirar)

    decisoes.append({
        "Manter": manter,
        "Retirar da lista": retirar
    })

# Usa a base atual sem alterar os arquivos originais.
base_revisada = base_provisoria[
    ~base_provisoria["arquivo"].isin(duplicatas_visuais)
].copy()

display(pd.DataFrame(decisoes))

print("Duplicatas visuais retiradas:", len(duplicatas_visuais))
print("Imagens na base revisada:", len(base_revisada))

display(
    base_revisada.groupby("classe")
    .size()
    .rename("Quantidade")
    .to_frame()
)

In [ ]:
resumo = base_revisada.groupby("classe").size().to_frame("Quantidade")
resumo.loc["TOTAL"] = resumo["Quantidade"].sum()

display(resumo)